# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Pathing
from pathlib import Path

# Modelling
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score, make_scorer
import joblib # For persistence

## Load Processed Feature Dataset for Modelling

In [ ]:
df_model = pd.read_csv("../data/processed/stage1_features.csv")
print("Loaded dataset shape:", df_model.shape)
display(df_model.head())

## Train/Test Split

In [ ]:
df_train, _ = train_test_split(
    df_model, test_size=0.2, stratify=df_model["Label"], random_state=42
)

X_train = df_train.drop("Label", axis=1)
y_train = df_train["Label"]

attack_fraction = y_train.sum()/len(y_train)
print("Contamination fraction:", attack_fraction)

# Model Fit

## Isolation Forest Model

### Initialise Model

In [ ]:
model = Pipeline([
    ("scaler", RobustScaler()),
    ("iforest", IsolationForest(
        n_estimators=1000,
        max_samples=4000,
        contamination=0.167,
        random_state=42,
        n_jobs=-1
    ))
])

### Fit Model

In [ ]:
print("Fitting isolation forest...")
model.fit(X_train)
print("Model fitting complete.")

### Hyperparameter Experiments (Optional)

This section is only needed if you want to explore different hyperparameter configurations. The final model uses the configuration above.

In [ ]:
# Custom scorer: attack f1
def attack_f1(y_true, y_pred):
    y_pred_mapped = np.where(y_pred == -1, 1, 0)
    return f1_score(y_true, y_pred_mapped)
f1_attack_scorer = make_scorer(attack_f1)

search_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("iforest", IsolationForest(random_state=42, n_jobs=-1))
])

# Hyperparameter grid
param_dist = {
    "iforest__n_estimators": [1000, 1500, 2000],
    "iforest__max_samples": [4000, 6000, 8000],
    "iforest__contamination": [0.16, 0.17, 0.18]
}

# Randomised search
search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_dist,
    n_iter=27,
    cv=4,
    scoring=f1_attack_scorer,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# Fit search
print("Running hyperparameter search...")
search.fit(X_train, y_train)
print("Hyperparameter search complete.")

# Inspect results
print("Best hyperparameters:", search.best_params_)
print("Best F1 (attack) score:", search.best_score_)
results_df = pd.DataFrame(search.cv_results_)

# Model Persistence

In [ ]:
model_path = Path("../models/isolation_forest_pipeline.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}.")